In [1]:
# ============================================================
# PHASE 5A — CROSS-PHASE FROZEN RESULT SYNTHESIS
# ============================================================
#
# PURPOSE
# -------
# Build one machine-readable summary of the principal frozen
# findings from:
#
#   Phase 1 — similarity / conservation
#   Phase 2 — international heterogeneity
#   Phase 3 — expanded predictor validation
#   Phase 4 — hierarchical association, moderation,
#             and indirect-association analyses
#
# NO NEW MODELS.
# NO NEW HYPOTHESIS TESTS.
#
# This notebook begins the synthesis/manuscript stage.
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "cross_phase_synthesis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Frozen source files
# ============================================================

FILES = {

    # --------------------------------------------------------
    # Phase 1
    # --------------------------------------------------------

    "similarity_observed":
        RESULT_ROOT
        / "similarity_framework_v2"
        / "observed_similarity_metrics.csv",

    # --------------------------------------------------------
    # Phase 3
    # --------------------------------------------------------

    "expanded43_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core14_vs_expanded43_performance_summary.csv",

    "expanded43_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core14_vs_expanded43_architecture_summary.csv",

    "expanded46_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core17_vs_expanded46_performance_summary.csv",

    "expanded46_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core17_vs_expanded46_architecture_summary.csv",

    # --------------------------------------------------------
    # Phase 4
    # --------------------------------------------------------

    "stage1_summary":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage1_focal_coefficient_descriptive_summary.csv",

    "pooled_associations":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_random_effects_pooled_results.csv",

    "moderation":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_primary_moderation_frozen_classification.csv",

    "gini":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_gini_frozen_classification.csv",

    "indirect":
        RESULT_ROOT
        / "multilevel_analysis"
        / "mediation_indirect_random_effects_meta_analysis.csv",

    "indirect_system":
        RESULT_ROOT
        / "multilevel_analysis"
        / "mediation_cluster_bootstrap_system_results.csv",
}


# ============================================================
# 3. File audit
# ============================================================

print("=" * 120)
print("FROZEN SOURCE FILE AUDIT")
print("=" * 120)


for name, path in FILES.items():

    print(
        f"{name:28s}",
        "FOUND" if path.exists() else "MISSING",
        "|",
        path
    )


# ============================================================
# 4. Load only files that exist
#
# Some Phase-3 filenames may differ slightly depending on
# the exact save names used in the earlier notebook.
#
# We therefore fail clearly instead of silently guessing.
# ============================================================

loaded = {}


for name, path in FILES.items():

    if path.exists():

        loaded[
            name
        ] = pd.read_csv(
            path
        )


# ============================================================
# 5. Phase 4 synthesis
#
# These filenames are known/frozen from the final analysis.
# ============================================================

required_phase4 = [

    "stage1_summary",

    "pooled_associations",

    "moderation",

    "gini",

    "indirect",

    "indirect_system",
]


for name in required_phase4:

    if name not in loaded:

        raise FileNotFoundError(
            f"Required frozen Phase-4 file missing: "
            f"{FILES[name]}"
        )


stage1 = loaded[
    "stage1_summary"
]


pooled = loaded[
    "pooled_associations"
]


moderation = loaded[
    "moderation"
]


gini = loaded[
    "gini"
]


indirect = loaded[
    "indirect"
]


indirect_system = loaded[
    "indirect_system"
]


# ============================================================
# 6. High-level frozen claim table
# ============================================================

claim_rows = []


# ------------------------------------------------------------
# Phase 4A/B — sign conservation
# ------------------------------------------------------------

for term in [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",
]:

    for outcome in sorted(
        stage1[
            "Outcome"
        ]
        .unique()
    ):

        row = (

            stage1[
                (
                    stage1[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage1[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
        )


        if len(
            row
        ) != 1:

            continue


        r = row.iloc[
            0
        ]


        claim_rows.append(
            {

                "Phase":
                    "4",

                "Analysis":
                    "Stage1 association",

                "Outcome":
                    outcome,

                "Feature":
                    term,

                "Metric":
                    "Sign consistency",

                "Value":
                    max(
                        float(
                            r[
                                "PositivePct"
                            ]
                        ),
                        float(
                            r[
                                "NegativePct"
                            ]
                        ),
                    ),

                "Direction":
                    (
                        "Positive"
                        if float(
                            r[
                                "PositivePct"
                            ]
                        )
                        >
                        float(
                            r[
                                "NegativePct"
                            ]
                        )
                        else
                        "Negative"
                    ),

                "Status":
                    "Frozen",
            }
        )


# ------------------------------------------------------------
# Phase 4C — pooled coefficients / heterogeneity
# ------------------------------------------------------------

for _, row in pooled.iterrows():

    claim_rows.append(
        {

            "Phase":
                "4",

            "Analysis":
                "Random-effects association",

            "Outcome":
                row[
                    "Outcome"
                ],

            "Feature":
                row[
                    "Term"
                ],

            "Metric":
                "PooledBeta",

            "Value":
                row[
                    "PooledBeta"
                ],

            "Direction":
                (
                    "Positive"
                    if row[
                        "PooledBeta"
                    ]
                    >
                    0
                    else
                    "Negative"
                ),

            "Status":
                "Frozen",
        }
    )


    claim_rows.append(
        {

            "Phase":
                "4",

            "Analysis":
                "Random-effects heterogeneity",

            "Outcome":
                row[
                    "Outcome"
                ],

            "Feature":
                row[
                    "Term"
                ],

            "Metric":
                "I2_Pct",

            "Value":
                row[
                    "I2_Pct"
                ],

            "Direction":
                np.nan,

            "Status":
                "Frozen",
        }
    )


# ------------------------------------------------------------
# Phase 4D/E — moderation classifications
# ------------------------------------------------------------

for _, row in moderation.iterrows():

    claim_rows.append(
        {

            "Phase":
                "4",

            "Analysis":
                "GDP/region moderation",

            "Outcome":
                row[
                    "Outcome"
                ],

            "Feature":
                row[
                    "Term"
                ],

            "Metric":
                row[
                    "Test"
                ],

            "Value":
                row[
                    "Estimate"
                ],

            "Direction":
                (
                    np.nan
                    if pd.isna(
                        row[
                            "Estimate"
                        ]
                    )
                    else
                    (
                        "Positive"
                        if row[
                            "Estimate"
                        ]
                        >
                        0
                        else
                        "Negative"
                    )
                ),

            "Status":
                row[
                    "FinalClassification"
                ],
        }
    )


# ------------------------------------------------------------
# Gini
# ------------------------------------------------------------

for _, row in gini.iterrows():

    claim_rows.append(
        {

            "Phase":
                "4",

            "Analysis":
                "Gini incremental moderation",

            "Outcome":
                row[
                    "Outcome"
                ],

            "Feature":
                row[
                    "Term"
                ],

            "Metric":
                "Gini | GDP + region",

            "Value":
                row[
                    "GiniBeta"
                ],

            "Direction":
                (
                    "Positive"
                    if row[
                        "GiniBeta"
                    ]
                    >
                    0
                    else
                    "Negative"
                ),

            "Status":
                row[
                    "FinalClassification"
                ],
        }
    )


# ------------------------------------------------------------
# Indirect association
# ------------------------------------------------------------

indirect_row = indirect.iloc[
    0
]


claim_rows.append(
    {

        "Phase":
            "4",

        "Analysis":
            "Indirect association",

        "Outcome":
            "BEINGBULLIED",

        "Feature":
            "DISCLIMA -> BELONG",

        "Metric":
            "PooledIndirect",

        "Value":
            indirect_row[
                "PooledIndirect"
            ],

        "Direction":
            "Negative",

        "Status":
            "Exploratory_Frozen",
    }
)


claim_rows.append(
    {

        "Phase":
            "4",

        "Analysis":
            "Indirect association",

        "Outcome":
            "BEINGBULLIED",

        "Feature":
            "DISCLIMA -> BELONG",

        "Metric":
            "SystemsBootstrapCIEntirelyBelowZero",

        "Value":
            int(
                indirect_system[
                    "CIEntirelyBelowZero"
                ]
                .sum()
            ),

        "Direction":
            "Negative",

        "Status":
            "Exploratory_Frozen",
    }
)


claims = pd.DataFrame(
    claim_rows
)


# ============================================================
# 7. Phase-4 freeze summary
# ============================================================

phase4_summary = pd.DataFrame(
    [

        {
            "Finding":
                "Stage1 models",

            "Value":
                374,
        },

        {
            "Finding":
                "Focal coefficients",

            "Value":
                1122,
        },

        {
            "Finding":
                "Robust GDP/region moderation tests",

            "Value":
                int(
                    (
                        moderation[
                            "FinalClassification"
                        ]
                        ==
                        "Robust_FDR"
                    )
                    .sum()
                ),
        },

        {
            "Finding":
                "Borderline GDP/region moderation tests",

            "Value":
                int(
                    (
                        moderation[
                            "FinalClassification"
                        ]
                        ==
                        "Borderline_Sensitivity"
                    )
                    .sum()
                ),
        },

        {
            "Finding":
                "Independent Gini FDR findings",

            "Value":
                int(
                    gini[
                        "FDR05"
                    ]
                    .sum()
                ),
        },

        {
            "Finding":
                "Indirect-association systems",

            "Value":
                len(
                    indirect_system
                ),
        },

        {
            "Finding":
                "Indirect point estimate negative",

            "Value":
                int(
                    (
                        indirect_system[
                            "Indirect"
                        ]
                        <
                        0
                    )
                    .sum()
                ),
        },

        {
            "Finding":
                "Indirect bootstrap CI below zero",

            "Value":
                int(
                    indirect_system[
                        "CIEntirelyBelowZero"
                    ]
                    .sum()
                ),
        },

        {
            "Finding":
                "Pooled indirect association",

            "Value":
                float(
                    indirect_row[
                        "PooledIndirect"
                    ]
                ),
        },

        {
            "Finding":
                "Pooled indirect CI low",

            "Value":
                float(
                    indirect_row[
                        "CI95_Low"
                    ]
                ),
        },

        {
            "Finding":
                "Pooled indirect CI high",

            "Value":
                float(
                    indirect_row[
                        "CI95_High"
                    ]
                ),
        },

        {
            "Finding":
                "Indirect I2",

            "Value":
                float(
                    indirect_row[
                        "I2_Pct"
                    ]
                ),
        },
    ]
)


# ============================================================
# 8. Save synthesis
# ============================================================

claims.to_csv(
    OUT_DIR
    / "frozen_claim_registry.csv",
    index=False
)


phase4_summary.to_csv(
    OUT_DIR
    / "phase4_frozen_summary.csv",
    index=False
)


# ============================================================
# 9. Print Phase-4 final summary
# ============================================================

print("\n")
print("=" * 120)
print("PHASE 4 FINAL FROZEN SUMMARY")
print("=" * 120)


print(
    phase4_summary
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 10. Moderation status counts
# ============================================================

print("\n")
print("=" * 120)
print("MODERATION STATUS COUNTS")
print("=" * 120)


print(
    moderation[
        "FinalClassification"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 11. Indirect-association freeze
# ============================================================

print("\n")
print("=" * 120)
print("INDIRECT-ASSOCIATION FREEZE")
print("=" * 120)


print(
    indirect[
        [
            "N_Systems",
            "PooledIndirect",
            "ModifiedKH_SE",
            "CI95_Low",
            "CI95_High",
            "Tau2",
            "I2_Pct",
            "PredictionInterval_Low",
            "PredictionInterval_High",
            "N_PointIndirectNegative",
            "N_BootstrapCIEntirelyBelowZero",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 120)
print("PHASE 5A SYNTHESIS REGISTRY CREATED")
print("=" * 120)


print(
    "Frozen claim rows:",
    len(
        claims
    )
)


print(
    "\nNext step: integrate the frozen Phase 1–4 results "
    "into the final manuscript argument and figure plan."
)

FROZEN SOURCE FILE AUDIT
similarity_observed          MISSING | ..\results\similarity_framework_v2\observed_similarity_metrics.csv
expanded43_performance       FOUND | ..\results\expanded_predictor_analysis\core14_vs_expanded43_performance_summary.csv
expanded43_architecture      MISSING | ..\results\expanded_predictor_analysis\core14_vs_expanded43_architecture_summary.csv
expanded46_performance       FOUND | ..\results\expanded_predictor_analysis\core17_vs_expanded46_performance_summary.csv
expanded46_architecture      MISSING | ..\results\expanded_predictor_analysis\core17_vs_expanded46_architecture_summary.csv
stage1_summary               FOUND | ..\results\multilevel_analysis\stage1_focal_coefficient_descriptive_summary.csv
pooled_associations          FOUND | ..\results\multilevel_analysis\stage2_random_effects_pooled_results.csv
moderation                   FOUND | ..\results\multilevel_analysis\stage2_primary_moderation_frozen_classification.csv
gini                         FOUN

In [2]:
# ============================================================
# PHASE 5A1 — LOCATE FROZEN PHASE 1 / PHASE 3 SOURCE FILES
# ============================================================
#
# NO ANALYSIS.
# NO FILE MODIFICATION.
#
# Simply inspect the existing frozen result directories so we
# can map the exact filenames into the synthesis notebook.
# ============================================================

from pathlib import Path
import pandas as pd


RESULT_ROOT = Path("../results")

SEARCH_DIRS = {

    "PHASE 1 — SIMILARITY":
        RESULT_ROOT / "similarity_framework_v2",

    "PHASE 3 — EXPANDED PREDICTORS":
        RESULT_ROOT / "expanded_predictor_analysis",
}


# ============================================================
# Helper: safely preview CSV structure
# ============================================================

def preview_csv(path):

    try:

        df = pd.read_csv(
            path,
            nrows=5
        )

        return {
            "rows_previewed": len(df),
            "columns": df.columns.tolist(),
        }

    except Exception as error:

        return {
            "error": str(error)
        }


# ============================================================
# 1. List likely relevant files
# ============================================================

KEYWORDS = [

    "similar",
    "observ",
    "metric",
    "summary",
    "architecture",
    "preserv",
    "core14",
    "expanded43",
    "core17",
    "expanded46",
]


for section, directory in SEARCH_DIRS.items():

    print("\n")
    print("=" * 120)
    print(section)
    print("=" * 120)

    print(
        "Directory:",
        directory.resolve()
    )

    if not directory.exists():

        print("DIRECTORY DOES NOT EXIST")

        continue


    csv_files = sorted(
        directory.glob("*.csv")
    )


    relevant = [

        path

        for path in csv_files

        if any(
            keyword in path.name.lower()
            for keyword in KEYWORDS
        )
    ]


    print(
        "\nRelevant CSV candidates:",
        len(relevant)
    )


    for path in relevant:

        print("\n" + "-" * 120)

        print(
            "FILE:",
            path.name
        )

        info = preview_csv(
            path
        )

        if "error" in info:

            print(
                "ERROR:",
                info["error"]
            )

        else:

            print(
                "COLUMNS:"
            )

            print(
                info[
                    "columns"
                ]
            )


# ============================================================
# 2. Specifically search Phase 3 for the two architecture
#    result families
# ============================================================

phase3_dir = (
    RESULT_ROOT
    / "expanded_predictor_analysis"
)


print("\n")
print("=" * 120)
print("PHASE 3 — CORE14 / EXPANDED43 CANDIDATES")
print("=" * 120)


for path in sorted(
    phase3_dir.glob("*.csv")
):

    name = path.name.lower()

    if (
        "core14" in name
        or
        "expanded43" in name
    ):

        print(
            path.name
        )


print("\n")
print("=" * 120)
print("PHASE 3 — CORE17 / EXPANDED46 CANDIDATES")
print("=" * 120)


for path in sorted(
    phase3_dir.glob("*.csv")
):

    name = path.name.lower()

    if (
        "core17" in name
        or
        "expanded46" in name
    ):

        print(
            path.name
        )


# ============================================================
# 3. Phase 1 complete CSV filenames
# ============================================================

phase1_dir = (
    RESULT_ROOT
    / "similarity_framework_v2"
)


print("\n")
print("=" * 120)
print("PHASE 1 — ALL CSV FILENAMES")
print("=" * 120)


for path in sorted(
    phase1_dir.glob("*.csv")
):

    print(
        path.name
    )


print("\n")
print("=" * 120)
print("PHASE 5A1 FILE DISCOVERY COMPLETE")
print("=" * 120)



PHASE 1 — SIMILARITY
Directory: C:\Users\adhiv\OneDrive\Desktop\EduMindsResearch\EduMinds-Research\results\similarity_framework_v2

Relevant CSV candidates: 7

------------------------------------------------------------------------------------------------------------------------
FILE: metric_pairwise_ordering_agreement.csv
COLUMNS:
['Outcome', 'Profile', 'Metric1', 'Metric2', 'PairwiseOrderingSpearman', 'P']

------------------------------------------------------------------------------------------------------------------------
FILE: observed_pairwise_metrics.csv
COLUMNS:
['Outcome', 'Profile', 'System1', 'System2', 'Cosine', 'Aitchison', 'Hellinger', 'Spearman']

------------------------------------------------------------------------------------------------------------------------
FILE: similarity_framework_compact.csv
COLUMNS:
['Outcome', 'Profile', 'MeanCosine_Observed', 'MeanCosine_NullMean', 'MeanCosine_P', 'MeanAitchison_Observed', 'MeanAitchison_NullMean', 'MeanAitchison_P',

In [3]:
# ============================================================
# PHASE 5A2 — VALIDATE PHASE 1/3/4 + DISCOVER PHASE 2 SOURCES
# ============================================================
#
# NO NEW ANALYSIS.
# NO MODEL FITTING.
# NO MODIFICATION OF FROZEN RESULTS.
#
# PURPOSE
# -------
# 1. Correct the Phase-1 and Phase-3 filenames.
# 2. Validate the main frozen Phase-1 / Phase-3 / Phase-4
#    source tables.
# 3. Locate the exact Phase-2 files needed for the final
#    cross-phase synthesis.
#
# We deliberately discover Phase-2 files rather than guessing
# their filenames.
# ============================================================


from pathlib import Path
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


# ============================================================
# 2. Exact frozen files now resolved
# ============================================================

FILES = {

    # --------------------------------------------------------
    # PHASE 1
    # --------------------------------------------------------

    "similarity_compact":
        RESULT_ROOT
        / "similarity_framework_v2"
        / "similarity_framework_compact.csv",


    # --------------------------------------------------------
    # PHASE 3 — Core14 vs Expanded43
    # --------------------------------------------------------

    "expanded43_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core14_vs_expanded43_performance_summary.csv",

    "expanded43_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core14_preservation_summary_expanded43.csv",

    "expanded43_global":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded43_global_attribution_summary.csv",

    "expanded43_new":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded43_new_feature_entrants.csv",


    # --------------------------------------------------------
    # PHASE 3 — Core17 vs Expanded46
    # --------------------------------------------------------

    "expanded46_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core17_vs_expanded46_performance_summary.csv",

    "expanded46_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core17_preservation_summary_expanded46.csv",

    "expanded46_global":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_global_attribution_summary.csv",

    "expanded46_new":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_new29_entrants.csv",


    # --------------------------------------------------------
    # PHASE 3 — literal Full47 parent sensitivity
    # --------------------------------------------------------

    "full47_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_vs_full47_parent_performance_summary.csv",

    "full47_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_vs_full47_parent_architecture_summary.csv",

    "parent_importance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "pa006_parent_predictor_importance_summary.csv",


    # --------------------------------------------------------
    # PHASE 4
    # --------------------------------------------------------

    "stage1_summary":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage1_focal_coefficient_descriptive_summary.csv",

    "pooled_associations":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_random_effects_pooled_results.csv",

    "moderation":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_primary_moderation_frozen_classification.csv",

    "gini":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_gini_frozen_classification.csv",

    "indirect":
        RESULT_ROOT
        / "multilevel_analysis"
        / "mediation_indirect_random_effects_meta_analysis.csv",

    "indirect_system":
        RESULT_ROOT
        / "multilevel_analysis"
        / "mediation_cluster_bootstrap_system_results.csv",
}


# ============================================================
# 3. Validate exact frozen files
# ============================================================

print("\n")
print("=" * 125)
print("CORRECTED PHASE 1 / 3 / 4 SOURCE AUDIT")
print("=" * 125)


missing = []


for name, path in FILES.items():

    exists = path.exists()


    print(
        f"{name:28s}",
        "FOUND" if exists else "MISSING",
        "|",
        path
    )


    if not exists:

        missing.append(
            name
        )


if len(
    missing
) > 0:

    raise FileNotFoundError(
        "Unexpected missing frozen files: "
        +
        ", ".join(
            missing
        )
    )


print(
    "\nAll corrected Phase 1 / 3 / 4 "
    "frozen source files were found."
)


# ============================================================
# 4. Load and validate important schemas
# ============================================================

tables = {

    name:
        pd.read_csv(
            path
        )

    for name, path
    in FILES.items()
}


EXPECTED_COLUMNS = {

    "similarity_compact": [
        "Outcome",
        "Profile",
        "MeanCosine_Observed",
        "MeanAitchison_Observed",
        "MeanHellinger_Observed",
        "MeanSpearman_Observed",
        "KendallW_Observed",
    ],

    "expanded43_performance": [
        "Outcome",
        "N_Systems",
        "Mean_Core",
        "Mean_Expanded",
        "Mean_Delta",
        "N_ExpandedBetter",
    ],

    "expanded43_architecture": [
        "Outcome",
        "N_Systems",
        "Mean_Core14_AttributionMass",
        "Mean_Core14_RenormCosine",
        "Mean_Core14_RankSpearman",
        "BELONG_MeanRank",
    ],

    "expanded46_performance": [
        "Outcome",
        "N_Systems",
        "Mean_Core",
        "Mean_Expanded",
        "Mean_Delta",
        "N_ExpandedBetter",
    ],

    "expanded46_architecture": [
        "Outcome",
        "N_Systems",
        "Mean_Core17_AttributionMass",
        "Mean_Core17_RenormCosine",
        "Mean_Core17_RankSpearman",
        "BELONG_MeanRank",
        "CoreAchievementShare",
        "ExpandedAchievementShare",
    ],

    "moderation": [
        "Outcome",
        "Term",
        "Test",
        "FinalClassification",
    ],

    "indirect": [
        "N_Systems",
        "PooledIndirect",
        "CI95_Low",
        "CI95_High",
        "I2_Pct",
    ],
}


for name, required_columns in (
    EXPECTED_COLUMNS.items()
):

    missing_columns = [

        column

        for column
        in required_columns

        if column
        not in tables[
            name
        ].columns
    ]


    if len(
        missing_columns
    ) > 0:

        raise RuntimeError(
            f"{name} missing expected columns: "
            f"{missing_columns}"
        )


print(
    "Key frozen schemas validated successfully."
)


# ============================================================
# 5. Quick Phase-1 verification
# ============================================================

similarity = tables[
    "similarity_compact"
]


print("\n")
print("=" * 125)
print("PHASE 1 SOURCE CHECK")
print("=" * 125)


print(
    "Rows:",
    len(
        similarity
    )
)


print(
    "Outcomes:",
    sorted(
        similarity[
            "Outcome"
        ]
        .unique()
    )
)


print(
    "Profiles:",
    sorted(
        similarity[
            "Profile"
        ]
        .unique()
    )
)


print("\nCompact similarity table:")


print(
    similarity
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 6. Quick Phase-3 verification
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 3 SOURCE CHECK — CORE14 VS EXPANDED43")
print("=" * 125)


print(
    tables[
        "expanded43_architecture"
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 125)
print("PHASE 3 SOURCE CHECK — CORE17 VS EXPANDED46")
print("=" * 125)


print(
    tables[
        "expanded46_architecture"
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 7. PHASE 2 DISCOVERY
#
# Search recursively across results for files plausibly tied
# to the frozen international-heterogeneity analysis.
# ============================================================

PHASE2_KEYWORDS = [

    "heterogeneity",

    "pca",

    "compositional",

    "macro",

    "permanova",

    "dispersion",

    "cluster",

    "kmeans",

    "reweight",

    "subtype",

    "context",

    "external",

    "gdp",

    "gini",

    "region",
]


# Exclude directories clearly belonging to other phases.
EXCLUDED_DIR_PARTS = {

    "similarity_framework_v2",

    "expanded_predictor_analysis",

    "multilevel_analysis",

    "cross_phase_synthesis",

    "achievement_pv",
}


candidate_paths = []


for path in RESULT_ROOT.rglob(
    "*.csv"
):

    relative_parts = set(
        path.relative_to(
            RESULT_ROOT
        ).parts
    )


    if len(
        relative_parts
        &
        EXCLUDED_DIR_PARTS
    ) > 0:

        continue


    searchable = str(
        path.relative_to(
            RESULT_ROOT
        )
    ).lower()


    if any(
        keyword
        in searchable

        for keyword
        in PHASE2_KEYWORDS
    ):

        candidate_paths.append(
            path
        )


candidate_paths = sorted(
    set(
        candidate_paths
    )
)


# ============================================================
# 8. Print Phase-2 candidate files + schemas
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 2 — FROZEN RESULT FILE CANDIDATES")
print("=" * 125)


print(
    "Candidate files:",
    len(
        candidate_paths
    )
)


phase2_discovery_rows = []


for path in candidate_paths:

    relative_path = path.relative_to(
        RESULT_ROOT
    )


    try:

        preview = pd.read_csv(
            path,
            nrows=5
        )


        columns = preview.columns.tolist()


        phase2_discovery_rows.append(
            {

                "RelativePath":
                    str(
                        relative_path
                    ),

                "Filename":
                    path.name,

                "Columns":
                    "|".join(
                        columns
                    ),

                "Readable":
                    True,
            }
        )


        print("\n" + "-" * 125)

        print(
            "FILE:",
            relative_path
        )


        print(
            "COLUMNS:"
        )


        print(
            columns
        )


    except Exception as error:

        phase2_discovery_rows.append(
            {

                "RelativePath":
                    str(
                        relative_path
                    ),

                "Filename":
                    path.name,

                "Columns":
                    "",

                "Readable":
                    False,

                "Error":
                    str(
                        error
                    ),
            }
        )


        print("\n" + "-" * 125)

        print(
            "FILE:",
            relative_path
        )


        print(
            "ERROR:",
            error
        )


# ============================================================
# 9. Also inspect likely Phase-2 directories even if their
# filenames do not contain our keywords.
# ============================================================

print("\n")
print("=" * 125)
print("TOP-LEVEL RESULT DIRECTORIES")
print("=" * 125)


top_level_dirs = sorted(
    [

        path

        for path
        in RESULT_ROOT.iterdir()

        if path.is_dir()
    ]
)


for directory in top_level_dirs:

    print(
        directory.name
    )


# ============================================================
# 10. Save discovery manifest
# ============================================================

phase2_discovery = pd.DataFrame(
    phase2_discovery_rows
)


DISCOVERY_OUT = (
    RESULT_ROOT
    / "cross_phase_synthesis"
)


DISCOVERY_OUT.mkdir(
    parents=True,
    exist_ok=True
)


phase2_discovery.to_csv(
    DISCOVERY_OUT
    / "phase2_source_discovery.csv",
    index=False
)


# ============================================================
# 11. Status
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 5A2 COMPLETE")
print("=" * 125)


print(
    "Phase 1 exact source resolved:"
)


print(
    FILES[
        "similarity_compact"
    ]
)


print(
    "\nPhase 3 exact architecture sources resolved:"
)


print(
    FILES[
        "expanded43_architecture"
    ]
)


print(
    FILES[
        "expanded46_architecture"
    ]
)


print(
    "\nPhase 4 sources remain fully frozen and validated."
)


print(
    "\nNext: map the Phase-2 candidate files into the final "
    "Phase 1–4 synthesis registry."
)



CORRECTED PHASE 1 / 3 / 4 SOURCE AUDIT
similarity_compact           FOUND | ..\results\similarity_framework_v2\similarity_framework_compact.csv
expanded43_performance       FOUND | ..\results\expanded_predictor_analysis\core14_vs_expanded43_performance_summary.csv
expanded43_architecture      FOUND | ..\results\expanded_predictor_analysis\core14_preservation_summary_expanded43.csv
expanded43_global            FOUND | ..\results\expanded_predictor_analysis\expanded43_global_attribution_summary.csv
expanded43_new               FOUND | ..\results\expanded_predictor_analysis\expanded43_new_feature_entrants.csv
expanded46_performance       FOUND | ..\results\expanded_predictor_analysis\core17_vs_expanded46_performance_summary.csv
expanded46_architecture      FOUND | ..\results\expanded_predictor_analysis\core17_preservation_summary_expanded46.csv
expanded46_global            FOUND | ..\results\expanded_predictor_analysis\expanded46_global_attribution_summary.csv
expanded46_new            

In [4]:
# ============================================================
# PHASE 5B — FINAL PHASE 1–4 FROZEN SYNTHESIS REGISTRY
# ============================================================
#
# PURPOSE
# -------
# Create one authoritative, machine-readable registry of the
# frozen findings from:
#
#   Phase 1 — cross-system conservation / similarity
#   Phase 2 — structured international heterogeneity
#   Phase 3 — expanded predictor-space validation
#   Phase 4 — hierarchical association, moderation, and
#             indirect-association analysis
#
#
# IMPORTANT
# ---------
#
# NO NEW MODELS.
# NO NEW PERMUTATIONS.
# NO NEW HYPOTHESIS TESTS.
# NO NEW CLUSTERING.
#
# Every result below is read from an already-frozen output.
#
# Derived quantities are purely descriptive summaries of
# those frozen outputs.
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "cross_phase_synthesis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Exact frozen source registry
# ============================================================

FILES = {

    # ========================================================
    # PHASE 1
    # ========================================================

    "p1_similarity":
        RESULT_ROOT
        / "similarity_framework_v2"
        / "similarity_framework_compact.csv",


    # ========================================================
    # PHASE 2A — compositional heterogeneity
    # ========================================================

    "p2_pca":
        RESULT_ROOT
        / "international_heterogeneity"
        / "pca_explained_variance.csv",

    "p2_cross_outcome":
        RESULT_ROOT
        / "international_heterogeneity"
        / "cross_outcome_distance_agreement.csv",

    "p2_center_distance":
        RESULT_ROOT
        / "international_heterogeneity"
        / "between_outcome_center_distances.csv",

    "p2_distinctiveness":
        RESULT_ROOT
        / "international_heterogeneity"
        / "system_aitchison_distinctiveness.csv",


    # ========================================================
    # PHASE 2B — macro context
    # ========================================================

    "p2_macro_tests":
        RESULT_ROOT
        / "macro_context"
        / "macro_multivariate_tests.csv",

    "p2_macro_dispersion":
        RESULT_ROOT
        / "macro_context"
        / "macro_dispersion_tests.csv",

    "p2_macro_coverage":
        RESULT_ROOT
        / "macro_context"
        / "macro_metadata_coverage.csv",


    # ========================================================
    # PHASE 2C — cluster interpretation
    # ========================================================

    "p2_cluster_robustness":
        RESULT_ROOT
        / "cluster_interpretation"
        / "focal_cluster_solution_robustness.csv",

    "p2_cluster_agreement":
        RESULT_ROOT
        / "cluster_interpretation"
        / "focal_cluster_cross_solution_agreement.csv",

    "p2_cluster_macro":
        RESULT_ROOT
        / "cluster_interpretation"
        / "focal_cluster_macro_summary.csv",


    # ========================================================
    # PHASE 2D — subtype reweighting
    # ========================================================

    "p2_reweight_cluster":
        RESULT_ROOT
        / "subtype_reweighting"
        / "reweighting_cluster_diagnostics.csv",

    "p2_reweight_intensity":
        RESULT_ROOT
        / "subtype_reweighting"
        / "reweighting_intensity_macro_correlations.csv",

    "p2_reweight_global_macro":
        RESULT_ROOT
        / "subtype_reweighting"
        / "global_reweighting_macro_tests.csv",

    "p2_reweight_feature_macro":
        RESULT_ROOT
        / "subtype_reweighting"
        / "feature_reweighting_macro_tests.csv",

    "p2_reweight_separation":
        RESULT_ROOT
        / "subtype_reweighting_k2_final"
        / "reweight_k2_separation_summary.csv",

    "p2_reweight_feature_separation":
        RESULT_ROOT
        / "subtype_reweighting_k2_final"
        / "reweight_k2_feature_separation.csv",

    "p2_reweight_subtype_separation":
        RESULT_ROOT
        / "subtype_reweighting_k2_final"
        / "reweight_k2_subtype_separation.csv",


    # ========================================================
    # PHASE 2E — targeted external context extension
    # ========================================================

    "p2_extended_context":
        RESULT_ROOT
        / "extended_context_patterns"
        / "extended_context_global_tests.csv",

    "p2_extended_context_adjusted":
        RESULT_ROOT
        / "extended_context_patterns"
        / "extended_context_adjusted_significant.csv",

    "p2_extended_context_repaired":
        RESULT_ROOT
        / "extended_context_patterns"
        / "repaired_high_priority_context_tests.csv",


    # ========================================================
    # PHASE 3A — Core14 vs Expanded43
    # ========================================================

    "p3_43_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core14_vs_expanded43_performance_summary.csv",

    "p3_43_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core14_preservation_summary_expanded43.csv",

    "p3_43_global":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded43_global_attribution_summary.csv",

    "p3_43_new":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded43_new_feature_entrants.csv",


    # ========================================================
    # PHASE 3B — Core17 vs Expanded46
    # ========================================================

    "p3_46_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core17_vs_expanded46_performance_summary.csv",

    "p3_46_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "core17_preservation_summary_expanded46.csv",

    "p3_46_global":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_global_attribution_summary.csv",

    "p3_46_new":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_new29_entrants.csv",


    # ========================================================
    # PHASE 3C — literal Full47 sensitivity
    # ========================================================

    "p3_47_performance":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_vs_full47_parent_performance_summary.csv",

    "p3_47_architecture":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "expanded46_vs_full47_parent_architecture_summary.csv",

    "p3_parent":
        RESULT_ROOT
        / "expanded_predictor_analysis"
        / "pa006_parent_predictor_importance_summary.csv",


    # ========================================================
    # PHASE 4A/B — system-specific associations
    # ========================================================

    "p4_stage1":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage1_focal_coefficient_descriptive_summary.csv",


    # ========================================================
    # PHASE 4C — random-effects associations
    # ========================================================

    "p4_pooled":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_random_effects_pooled_results.csv",


    # ========================================================
    # PHASE 4D/E — moderation
    # ========================================================

    "p4_moderation":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_primary_moderation_frozen_classification.csv",

    "p4_gini":
        RESULT_ROOT
        / "multilevel_analysis"
        / "stage2_gini_frozen_classification.csv",


    # ========================================================
    # PHASE 4F/G — indirect association
    # ========================================================

    "p4_indirect":
        RESULT_ROOT
        / "multilevel_analysis"
        / "mediation_indirect_random_effects_meta_analysis.csv",

    "p4_indirect_system":
        RESULT_ROOT
        / "multilevel_analysis"
        / "mediation_cluster_bootstrap_system_results.csv",
}


# ============================================================
# 3. Full source audit
# ============================================================

print("\n")
print("=" * 135)
print("FINAL FROZEN SOURCE AUDIT")
print("=" * 135)


missing = []


for key, path in FILES.items():

    exists = path.exists()


    print(
        f"{key:32s}",
        "FOUND" if exists else "MISSING",
        "|",
        path
    )


    if not exists:

        missing.append(
            key
        )


if missing:

    raise FileNotFoundError(
        "Missing frozen files: "
        +
        ", ".join(
            missing
        )
    )


print(
    "\nAll frozen Phase 1–4 synthesis sources found."
)


# ============================================================
# 4. Load
# ============================================================

T = {

    key:
        pd.read_csv(
            path
        )

    for key, path
    in FILES.items()
}


# ============================================================
# 5. Generic claim registry helper
# ============================================================

claims = []


def add_claim(
    phase,
    domain,
    outcome,
    feature,
    metric,
    value,
    status,
    source_key,
    note=None,
):

    claims.append(
        {

            "Phase":
                phase,

            "Domain":
                domain,

            "Outcome":
                outcome,

            "Feature":
                feature,

            "Metric":
                metric,

            "Value":
                value,

            "Status":
                status,

            "SourceKey":
                source_key,

            "SourceFile":
                str(
                    FILES[
                        source_key
                    ]
                    .relative_to(
                        RESULT_ROOT
                    )
                ),

            "Note":
                note,
        }
    )


# ============================================================
# ============================================================
# PHASE 1 — INTERNATIONAL CONSERVATION
# ============================================================
# ============================================================

similarity = T[
    "p1_similarity"
]


for _, row in similarity.iterrows():

    outcome = row[
        "Outcome"
    ]

    profile = row[
        "Profile"
    ]


    # --------------------------------------------------------
    # Primary Aitchison geometry
    # --------------------------------------------------------

    add_claim(
        phase="1",
        domain="Cross-system conservation",
        outcome=outcome,
        feature=profile,
        metric="MeanAitchison_Observed",
        value=row[
            "MeanAitchison_Observed"
        ],
        status="Primary_Frozen",
        source_key="p1_similarity",
        note=(
            "Lower distance indicates greater "
            "compositional similarity."
        ),
    )


    add_claim(
        phase="1",
        domain="Cross-system conservation",
        outcome=outcome,
        feature=profile,
        metric="MeanAitchison_NullMean",
        value=row[
            "MeanAitchison_NullMean"
        ],
        status="PermutationNull",
        source_key="p1_similarity",
    )


    add_claim(
        phase="1",
        domain="Cross-system conservation",
        outcome=outcome,
        feature=profile,
        metric="Aitchison_PermutationP",
        value=row[
            "MeanAitchison_P"
        ],
        status="Primary_Frozen",
        source_key="p1_similarity",
        note="10,000 predictor-label permutations.",
    )


    # --------------------------------------------------------
    # Cosine
    # --------------------------------------------------------

    add_claim(
        phase="1",
        domain="Cross-system conservation",
        outcome=outcome,
        feature=profile,
        metric="MeanCosine_Observed",
        value=row[
            "MeanCosine_Observed"
        ],
        status="Secondary_Frozen",
        source_key="p1_similarity",
    )


    # --------------------------------------------------------
    # Hellinger
    # --------------------------------------------------------

    add_claim(
        phase="1",
        domain="Cross-system conservation",
        outcome=outcome,
        feature=profile,
        metric="MeanHellinger_Observed",
        value=row[
            "MeanHellinger_Observed"
        ],
        status="Robustness_Frozen",
        source_key="p1_similarity",
    )


    # --------------------------------------------------------
    # Rank agreement
    # --------------------------------------------------------

    add_claim(
        phase="1",
        domain="Cross-system conservation",
        outcome=outcome,
        feature=profile,
        metric="KendallW_Observed",
        value=row[
            "KendallW_Observed"
        ],
        status="RankAgreement_Frozen",
        source_key="p1_similarity",
    )


# ============================================================
# ============================================================
# PHASE 2 — STRUCTURED HETEROGENEITY
# ============================================================
# ============================================================


# ============================================================
# 6. Compositional PCA
# ============================================================

pca = T[
    "p2_pca"
]


full14_pca = (

    pca[
        pca[
            "Profile"
        ]
        ==
        "Full14"
    ]
    .copy()
)


# First 3 PCs cumulative variance by outcome
for outcome in sorted(
    full14_pca[
        "Outcome"
    ]
    .unique()
):

    temp = (

        full14_pca[
            full14_pca[
                "Outcome"
            ]
            ==
            outcome
        ]
        .sort_values(
            "PC"
        )
    )


    pc3 = temp[
        temp[
            "PC"
        ]
        ==
        3
    ]


    if len(
        pc3
    ) == 1:

        add_claim(
            phase="2",
            domain="Compositional heterogeneity",
            outcome=outcome,
            feature="Full14",
            metric="First3PC_CumulativeVariance",
            value=float(
                pc3[
                    "CumulativeVariance"
                ]
                .iloc[
                    0
                ]
            ),
            status="Frozen",
            source_key="p2_pca",
            note=(
                "CLR compositional PCA; axes describe "
                "relative attribution reweighting."
            ),
        )


    # Components required for 80%
    components80 = temp[
        "ComponentsFor80Pct"
    ].dropna()


    if len(
        components80
    ) > 0:

        add_claim(
            phase="2",
            domain="Compositional heterogeneity",
            outcome=outcome,
            feature="Full14",
            metric="ComponentsFor80Pct",
            value=float(
                components80.iloc[
                    0
                ]
            ),
            status="Frozen",
            source_key="p2_pca",
        )


# ============================================================
# 7. Cross-outcome distance-structure agreement
# ============================================================

distance_agreement = T[
    "p2_cross_outcome"
]


distance_full14 = (

    distance_agreement[
        distance_agreement[
            "Profile"
        ]
        ==
        "Full14"
    ]
    .copy()
)


for _, row in distance_full14.iterrows():

    pair = (
        f"{row['Outcome1']} vs {row['Outcome2']}"
    )


    add_claim(
        phase="2",
        domain="Cross-outcome architecture",
        outcome=pair,
        feature="Full14",
        metric="PairwiseAitchisonDistanceSpearman",
        value=row[
            "PairwiseAitchisonDistanceSpearman"
        ],
        status="Frozen",
        source_key="p2_cross_outcome",
    )


# ============================================================
# 8. International-center distances between outcomes
# ============================================================

center_distance = T[
    "p2_center_distance"
]


center_full14 = (

    center_distance[
        center_distance[
            "Profile"
        ]
        ==
        "Full14"
    ]
    .copy()
)


for _, row in center_full14.iterrows():

    pair = (
        f"{row['Outcome1']} vs {row['Outcome2']}"
    )


    add_claim(
        phase="2",
        domain="Cross-outcome architecture",
        outcome=pair,
        feature="Full14",
        metric="CenterAitchisonDistance",
        value=row[
            "CenterAitchisonDistance"
        ],
        status="Frozen",
        source_key="p2_center_distance",
    )


# ============================================================
# 9. Macro PERMANOVA-like tests
# ============================================================

macro_tests = T[
    "p2_macro_tests"
]


for _, row in macro_tests.iterrows():

    add_claim(
        phase="2",
        domain="Macro-context heterogeneity",
        outcome=row[
            "Outcome"
        ],
        feature=(
            f"{row['Profile']} | "
            f"{row['MacroVariable']}"
        ),
        metric="R2",
        value=row[
            "R2"
        ],
        status=(
            "FDR_Significant"
            if (
                pd.notna(
                    row[
                        "FDR_Q"
                    ]
                )
                and
                row[
                    "FDR_Q"
                ]
                <
                0.05
            )
            else
            "Not_FDR_Significant"
        ),
        source_key="p2_macro_tests",
        note=(
            f"PermutationP={row['PermutationP']}; "
            f"FDR_Q={row['FDR_Q']}"
        ),
    )


# ============================================================
# 10. Dispersion checks
# ============================================================

dispersion = T[
    "p2_macro_dispersion"
]


for _, row in dispersion.iterrows():

    add_claim(
        phase="2",
        domain="Macro-context dispersion QA",
        outcome=row[
            "Outcome"
        ],
        feature=(
            f"{row['Profile']} | "
            f"{row['MacroVariable']}"
        ),
        metric="DispersionPermutationP",
        value=row[
            "DispersionPermutationP"
        ],
        status=(
            "PotentialDispersionDifference"
            if (
                pd.notna(
                    row[
                        "DispersionPermutationP"
                    ]
                )
                and
                row[
                    "DispersionPermutationP"
                ]
                <
                0.05
            )
            else
            "NoDispersionFlag"
        ),
        source_key="p2_macro_dispersion",
    )


# ============================================================
# 11. Focal cluster robustness
# ============================================================

cluster_robustness = T[
    "p2_cluster_robustness"
]


for _, row in cluster_robustness.iterrows():

    add_claim(
        phase="2",
        domain="Cluster robustness",
        outcome=row[
            "Representation"
        ],
        feature=row[
            "SolutionID"
        ],
        metric="BootstrapMeanARI",
        value=row[
            "BootstrapMeanARI"
        ],
        status="Exploratory_Frozen",
        source_key="p2_cluster_robustness",
        note=(
            f"K={row['K']}; "
            f"N={row['N_systems']}; "
            f"MeanMembershipProbability="
            f"{row['MeanMembershipProbability']}"
        ),
    )


# ============================================================
# 12. Subtype reweighting cluster diagnostics
# ============================================================

reweight_clusters = T[
    "p2_reweight_cluster"
]


for _, row in reweight_clusters.iterrows():

    add_claim(
        phase="2",
        domain="Subtype reweighting clusters",
        outcome="Four subtypes jointly",
        feature=row[
            "SolutionID"
        ],
        metric="StabilityMeanARI",
        value=row[
            "StabilityMeanARI"
        ],
        status="Exploratory_Frozen",
        source_key="p2_reweight_cluster",
        note=(
            f"Algorithm={row['Algorithm']}; "
            f"K={row['K']}; "
            f"Silhouette={row['Silhouette']}"
        ),
    )


# ============================================================
# 13. Reweighting intensity × macro context
# ============================================================

reweight_intensity = T[
    "p2_reweight_intensity"
]


for _, row in reweight_intensity.iterrows():

    add_claim(
        phase="2",
        domain="Subtype reweighting intensity",
        outcome="Four subtypes jointly",
        feature=row[
            "Validator"
        ],
        metric="SpearmanRho",
        value=row[
            "SpearmanRho"
        ],
        status=(
            "FDR_Significant"
            if (
                pd.notna(
                    row[
                        "FDR_Q"
                    ]
                )
                and
                row[
                    "FDR_Q"
                ]
                <
                0.05
            )
            else
            "Not_FDR_Significant"
        ),
        source_key="p2_reweight_intensity",
        note=(
            f"P={row['P']}; "
            f"FDR_Q={row['FDR_Q']}"
        ),
    )


# ============================================================
# 14. Final K2 reweighting separation decomposition
# ============================================================

reweight_sep = T[
    "p2_reweight_separation"
]


for _, row in reweight_sep.iterrows():

    add_claim(
        phase="2",
        domain="Subtype reweighting K2",
        outcome="Four subtypes jointly",
        feature=row[
            "Component"
        ],
        metric="PctOfTotalSquaredSeparation",
        value=row[
            "PctOfTotalSquaredSeparation"
        ],
        status="Exploratory_Frozen",
        source_key="p2_reweight_separation",
    )


feature_sep = T[
    "p2_reweight_feature_separation"
]


for _, row in feature_sep.iterrows():

    add_claim(
        phase="2",
        domain="Subtype reweighting K2",
        outcome="Four subtypes jointly",
        feature=row[
            "Feature"
        ],
        metric="FeaturePctOfTotalSquaredSeparation",
        value=row[
            "PctOfTotalSeparation"
        ],
        status="Exploratory_Frozen",
        source_key="p2_reweight_feature_separation",
    )


subtype_sep = T[
    "p2_reweight_subtype_separation"
]


for _, row in subtype_sep.iterrows():

    add_claim(
        phase="2",
        domain="Subtype reweighting K2",
        outcome=row[
            "Subtype"
        ],
        feature="Subtype block",
        metric="SubtypePctOfTotalSquaredSeparation",
        value=row[
            "PctOfTotalSeparation"
        ],
        status="Exploratory_Frozen",
        source_key="p2_reweight_subtype_separation",
    )


# ============================================================
# 15. Extended context robustness
# ============================================================

extended_adjusted = T[
    "p2_extended_context_adjusted"
]


extended_repaired = T[
    "p2_extended_context_repaired"
]


# Record every adjusted-context result
for _, row in extended_repaired.iterrows():

    add_claim(
        phase="2",
        domain="Extended external context",
        outcome=row[
            "Response"
        ],
        feature=row[
            "Validator"
        ],
        metric="PartialR2",
        value=row[
            "PartialR2"
        ],
        status=(
            "FDR_Significant"
            if (
                pd.notna(
                    row[
                        "FDR_Q"
                    ]
                )
                and
                row[
                    "FDR_Q"
                ]
                <
                0.05
            )
            else
            "Not_FDR_Significant"
        ),
        source_key="p2_extended_context_repaired",
        note=(
            f"Test={row['Test']}; "
            f"P={row['PermutationP']}; "
            f"FDR_Q={row['FDR_Q']}"
        ),
    )


# ============================================================
# ============================================================
# PHASE 3 — EXPANDED PREDICTOR VALIDATION
# ============================================================
# ============================================================


# ============================================================
# 16. Core14 vs Expanded43 performance
# ============================================================

perf43 = T[
    "p3_43_performance"
]


for _, row in perf43.iterrows():

    add_claim(
        phase="3",
        domain="Expanded43 predictive performance",
        outcome=row[
            "Outcome"
        ],
        feature="Core14 -> Expanded43",
        metric="MeanDeltaPrimaryMetric",
        value=row[
            "Mean_Delta"
        ],
        status="Primary_Frozen",
        source_key="p3_43_performance",
        note=(
            f"N={row['N_Systems']}; "
            f"ExpandedBetter={row['N_ExpandedBetter']}/"
            f"{row['N_Systems']}"
        ),
    )


# ============================================================
# 17. Core14 preservation inside Expanded43
# ============================================================

arch43 = T[
    "p3_43_architecture"
]


for _, row in arch43.iterrows():

    for metric in [

        "Mean_Core14_AttributionMass",

        "Mean_Core14_RenormCosine",

        "Mean_Core14_RankSpearman",

        "BELONG_MeanRank",

        "BELONG_Rank1_Count",

        "Mean_NewFeatures_in_Top10",

    ]:

        add_claim(
            phase="3",
            domain="Expanded43 architecture",
            outcome=row[
                "Outcome"
            ],
            feature="Core14",
            metric=metric,
            value=row[
                metric
            ],
            status="Primary_Frozen",
            source_key="p3_43_architecture",
        )


# ============================================================
# 18. Core17 vs Expanded46 performance
# ============================================================

perf46 = T[
    "p3_46_performance"
]


for _, row in perf46.iterrows():

    add_claim(
        phase="3",
        domain="Expanded46 predictive performance",
        outcome=row[
            "Outcome"
        ],
        feature="Core17 -> Expanded46",
        metric="MeanDeltaPrimaryMetric",
        value=row[
            "Mean_Delta"
        ],
        status="AchievementInclusive_Frozen",
        source_key="p3_46_performance",
        note=(
            f"N={row['N_Systems']}; "
            f"ExpandedBetter={row['N_ExpandedBetter']}/"
            f"{row['N_Systems']}"
        ),
    )


# ============================================================
# 19. Core17 preservation inside Expanded46
# ============================================================

arch46 = T[
    "p3_46_architecture"
]


for _, row in arch46.iterrows():

    for metric in [

        "Mean_Core17_AttributionMass",

        "Mean_Core17_RenormCosine",

        "Mean_Core17_RankSpearman",

        "BELONG_MeanRank",

        "CoreAchievementShare",

        "ExpandedAchievementShare",

        "READ_MeanRank",

        "MATH_MeanRank",

        "SCIE_MeanRank",

        "Mean_New29_in_Top10",

    ]:

        add_claim(
            phase="3",
            domain="Expanded46 architecture",
            outcome=row[
                "Outcome"
            ],
            feature="Core17",
            metric=metric,
            value=row[
                metric
            ],
            status="AchievementInclusive_Frozen",
            source_key="p3_46_architecture",
        )


# ============================================================
# 20. Full47 parent sensitivity
# ============================================================

perf47 = T[
    "p3_47_performance"
]


for _, row in perf47.iterrows():

    add_claim(
        phase="3",
        domain="Literal Full47 sensitivity",
        outcome=row[
            "Outcome"
        ],
        feature="PA006Q09TA",
        metric="MeanDeltaFull47",
        value=row[
            "Mean_Delta"
        ],
        status="Sensitivity_Frozen",
        source_key="p3_47_performance",
        note=(
            f"N={row['N_Systems']} systems only."
        ),
    )


parent = T[
    "p3_parent"
]


for _, row in parent.iterrows():

    add_claim(
        phase="3",
        domain="Literal Full47 sensitivity",
        outcome=row[
            "Outcome"
        ],
        feature="PA006Q09TA",
        metric="MeanSHAPShare",
        value=row[
            "MeanShare"
        ],
        status="Sensitivity_Frozen",
        source_key="p3_parent",
        note=(
            f"MeanRank={row['MeanRank']}; "
            f"Top10Count={row['Top10Count']}"
        ),
    )


# ============================================================
# ============================================================
# PHASE 4 — HIERARCHICAL ASSOCIATION / MODERATION
# ============================================================
# ============================================================


# ============================================================
# 21. Stage-1 sign consistency
# ============================================================

stage1 = T[
    "p4_stage1"
]


for _, row in stage1.iterrows():

    dominant_direction = (

        "Positive"

        if row[
            "PositivePct"
        ]
        >
        row[
            "NegativePct"
        ]

        else "Negative"
    )


    dominant_pct = max(

        row[
            "PositivePct"
        ],

        row[
            "NegativePct"
        ],
    )


    add_claim(
        phase="4",
        domain="System-specific adjusted association",
        outcome=row[
            "Outcome"
        ],
        feature=row[
            "Term"
        ],
        metric="DominantDirectionPct",
        value=dominant_pct,
        status="Frozen",
        source_key="p4_stage1",
        note=dominant_direction,
    )


# ============================================================
# 22. Random-effects pooled associations
# ============================================================

pooled = T[
    "p4_pooled"
]


for _, row in pooled.iterrows():

    for metric in [

        "PooledBeta",

        "I2_Pct",

        "PredictionInterval_Low",

        "PredictionInterval_High",

    ]:

        add_claim(
            phase="4",
            domain="Random-effects association",
            outcome=row[
                "Outcome"
            ],
            feature=row[
                "Term"
            ],
            metric=metric,
            value=row[
                metric
            ],
            status="Frozen",
            source_key="p4_pooled",
        )


# ============================================================
# 23. GDP / region moderation
# ============================================================

moderation = T[
    "p4_moderation"
]


for _, row in moderation.iterrows():

    add_claim(
        phase="4",
        domain="GDP/region moderation",
        outcome=row[
            "Outcome"
        ],
        feature=row[
            "Term"
        ],
        metric=row[
            "Test"
        ],
        value=row[
            "Estimate"
        ],
        status=row[
            "FinalClassification"
        ],
        source_key="p4_moderation",
        note=(
            f"P={row['P']}; "
            f"Q={row['Q_FDR30']}"
        ),
    )


# ============================================================
# 24. Incremental Gini
# ============================================================

gini = T[
    "p4_gini"
]


for _, row in gini.iterrows():

    add_claim(
        phase="4",
        domain="Gini moderation",
        outcome=row[
            "Outcome"
        ],
        feature=row[
            "Term"
        ],
        metric="Gini | GDP + region",
        value=row[
            "GiniBeta"
        ],
        status=row[
            "FinalClassification"
        ],
        source_key="p4_gini",
        note=(
            f"P={row['P']}; "
            f"Q={row['Q_FDR15']}"
        ),
    )


# ============================================================
# 25. Exploratory indirect association
# ============================================================

indirect = T[
    "p4_indirect"
]


indirect_system = T[
    "p4_indirect_system"
]


ir = indirect.iloc[
    0
]


for metric in [

    "PooledIndirect",

    "CI95_Low",

    "CI95_High",

    "I2_Pct",

    "PredictionInterval_Low",

    "PredictionInterval_High",

]:

    add_claim(
        phase="4",
        domain="Cross-sectional indirect association",
        outcome="BEINGBULLIED",
        feature="DISCLIMA -> BELONG",
        metric=metric,
        value=ir[
            metric
        ],
        status="Exploratory_Frozen",
        source_key="p4_indirect",
        note=(
            "Noncausal; cross-sectional indirect "
            "association only."
        ),
    )


add_claim(
    phase="4",
    domain="Cross-sectional indirect association",
    outcome="BEINGBULLIED",
    feature="DISCLIMA -> BELONG",
    metric="SystemsWithNegativeIndirect",
    value=int(
        (
            indirect_system[
                "Indirect"
            ]
            <
            0
        )
        .sum()
    ),
    status="Exploratory_Frozen",
    source_key="p4_indirect_system",
)


add_claim(
    phase="4",
    domain="Cross-sectional indirect association",
    outcome="BEINGBULLIED",
    feature="DISCLIMA -> BELONG",
    metric="SystemsBootstrapCIEntirelyBelowZero",
    value=int(
        indirect_system[
            "CIEntirelyBelowZero"
        ]
        .sum()
    ),
    status="Exploratory_Frozen",
    source_key="p4_indirect_system",
)


# ============================================================
# 26. Final registry
# ============================================================

claim_registry = pd.DataFrame(
    claims
)


claim_registry.insert(
    0,
    "ClaimID",
    [

        f"C{i:04d}"

        for i in range(
            1,
            len(
                claim_registry
            )
            +
            1
        )
    ]
)


claim_registry.to_csv(
    OUT_DIR
    / "final_frozen_claim_registry_phase1_to_phase4.csv",
    index=False
)


# ============================================================
# 27. Source manifest
# ============================================================

source_manifest = pd.DataFrame(
    [

        {

            "SourceKey":
                key,

            "RelativePath":
                str(
                    path.relative_to(
                        RESULT_ROOT
                    )
                ),

            "N_Rows":
                len(
                    T[
                        key
                    ]
                ),

            "N_Columns":
                len(
                    T[
                        key
                    ].columns
                ),
        }

        for key, path
        in FILES.items()
    ]
)


source_manifest.to_csv(
    OUT_DIR
    / "final_frozen_source_manifest.csv",
    index=False
)


# ============================================================
# 28. Phase-2 concise synthesis table
# ============================================================

phase2_summary_rows = []


# ------------------------------------------------------------
# PCA first-3-PC range
# ------------------------------------------------------------

pca3_claims = (

    claim_registry[
        (
            claim_registry[
                "Phase"
            ]
            ==
            "2"
        )
        &
        (
            claim_registry[
                "Metric"
            ]
            ==
            "First3PC_CumulativeVariance"
        )
    ]
)


phase2_summary_rows.append(
    {

        "Finding":
            "First 3 PCA components cumulative variance range",

        "Value":
            (
                f"{pca3_claims['Value'].min():.3f}"
                "–"
                f"{pca3_claims['Value'].max():.3f}"
            ),
    }
)


# ------------------------------------------------------------
# Components needed for 80%
# ------------------------------------------------------------

pca80_claims = (

    claim_registry[
        (
            claim_registry[
                "Phase"
            ]
            ==
            "2"
        )
        &
        (
            claim_registry[
                "Metric"
            ]
            ==
            "ComponentsFor80Pct"
        )
    ]
)


phase2_summary_rows.append(
    {

        "Finding":
            "Components required for 80% variance range",

        "Value":
            (
                f"{int(pca80_claims['Value'].min())}"
                "–"
                f"{int(pca80_claims['Value'].max())}"
            ),
    }
)


# ------------------------------------------------------------
# Strongest cross-outcome distance agreement
# ------------------------------------------------------------

cross_claims = (

    claim_registry[
        (
            claim_registry[
                "Phase"
            ]
            ==
            "2"
        )
        &
        (
            claim_registry[
                "Metric"
            ]
            ==
            "PairwiseAitchisonDistanceSpearman"
        )
    ]
)


strongest_cross = (

    cross_claims
    .sort_values(
        "Value",
        ascending=False
    )
    .iloc[
        0
    ]
)


phase2_summary_rows.append(
    {

        "Finding":
            "Strongest cross-outcome distance-structure agreement",

        "Value":
            (
                f"{strongest_cross['Outcome']} "
                f"(rho={strongest_cross['Value']:.3f})"
            ),
    }
)


# ------------------------------------------------------------
# Closest international centers
# ------------------------------------------------------------

center_claims = (

    claim_registry[
        (
            claim_registry[
                "Phase"
            ]
            ==
            "2"
        )
        &
        (
            claim_registry[
                "Metric"
            ]
            ==
            "CenterAitchisonDistance"
        )
    ]
)


closest_centers = (

    center_claims
    .sort_values(
        "Value"
    )
    .iloc[
        0
    ]
)


phase2_summary_rows.append(
    {

        "Finding":
            "Closest international outcome centers",

        "Value":
            (
                f"{closest_centers['Outcome']} "
                f"(Aitchison="
                f"{closest_centers['Value']:.3f})"
            ),
    }
)


# ------------------------------------------------------------
# Macro FDR significance
# ------------------------------------------------------------

macro_claims = (

    claim_registry[
        claim_registry[
            "Domain"
        ]
        ==
        "Macro-context heterogeneity"
    ]
)


phase2_summary_rows.append(
    {

        "Finding":
            "Macro tests surviving FDR",

        "Value":
            (
                f"{(macro_claims['Status'] == 'FDR_Significant').sum()}"
                "/"
                f"{len(macro_claims)}"
            ),
    }
)


# ------------------------------------------------------------
# Extended context
# ------------------------------------------------------------

extended_claims = (

    claim_registry[
        claim_registry[
            "Domain"
        ]
        ==
        "Extended external context"
    ]
)


phase2_summary_rows.append(
    {

        "Finding":
            "Extended-context tests surviving FDR",

        "Value":
            (
                f"{(extended_claims['Status'] == 'FDR_Significant').sum()}"
                "/"
                f"{len(extended_claims)}"
            ),
    }
)


phase2_summary = pd.DataFrame(
    phase2_summary_rows
)


phase2_summary.to_csv(
    OUT_DIR
    / "phase2_frozen_synthesis_summary.csv",
    index=False
)


# ============================================================
# 29. Cross-phase headline summary
# ============================================================

headline_rows = []


# ------------------------------------------------------------
# Phase 1
# ------------------------------------------------------------

full14_similarity = (

    similarity[
        similarity[
            "Profile"
        ]
        ==
        "Full14"
    ]
)


headline_rows.append(
    {

        "Phase":
            "1",

        "Headline":
            "Cross-system predictor architecture is strongly conserved.",

        "Evidence":
            (
                "Full14 mean cosine range "
                f"{full14_similarity['MeanCosine_Observed'].min():.3f}"
                "–"
                f"{full14_similarity['MeanCosine_Observed'].max():.3f}; "
                "all Aitchison permutation p=.0001."
            ),

        "InterpretationStatus":
            "Primary",
    }
)


# ------------------------------------------------------------
# Phase 2
# ------------------------------------------------------------

headline_rows.append(
    {

        "Phase":
            "2",

        "Headline":
            (
                "Conserved architecture contains structured "
                "continuous international heterogeneity."
            ),

        "Evidence":
            (
                "CLR-PCA is multidimensional; cross-outcome "
                "distance structures differ; macro context and "
                "subtype reweighting explain portions of this "
                "heterogeneity."
            ),

        "InterpretationStatus":
            "Primary + exploratory clustering",
    }
)


# ------------------------------------------------------------
# Phase 3
# ------------------------------------------------------------

headline_rows.append(
    {

        "Phase":
            "3",

        "Headline":
            (
                "The original predictor set is a stable backbone, "
                "not an exhaustive predictor set."
            ),

        "Evidence":
            (
                "Expanded predictors improve prediction while "
                "Core14/Core17 renormalized architecture remains "
                "near-identical."
            ),

        "InterpretationStatus":
            "Primary",
    }
)


# ------------------------------------------------------------
# Phase 4
# ------------------------------------------------------------

robust_mod = int(
    (
        moderation[
            "FinalClassification"
        ]
        ==
        "Robust_FDR"
    )
    .sum()
)


headline_rows.append(
    {

        "Phase":
            "4",

        "Headline":
            (
                "Dominant associations are directionally conserved "
                "but their magnitudes vary systematically across "
                "international context."
            ),

        "Evidence":
            (
                f"{robust_mod}/30 GDP/region moderator tests "
                "robust after FDR and sensitivity analysis; "
                f"indirect association negative in "
                f"{int((indirect_system['Indirect'] < 0).sum())}/"
                f"{len(indirect_system)} systems."
            ),

        "InterpretationStatus":
            (
                "Primary moderation; exploratory noncausal "
                "indirect association"
            ),
    }
)


headline_summary = pd.DataFrame(
    headline_rows
)


headline_summary.to_csv(
    OUT_DIR
    / "cross_phase_headline_summary.csv",
    index=False
)


# ============================================================
# 30. Print registry QA
# ============================================================

print("\n")
print("=" * 135)
print("FINAL CLAIM REGISTRY QA")
print("=" * 135)


print(
    "Total frozen claim rows:",
    len(
        claim_registry
    )
)


print("\nClaims by phase:")


print(
    claim_registry[
        "Phase"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print("\nClaims by status:")


print(
    claim_registry[
        "Status"
    ]
    .value_counts()
    .to_string()
)


print(
    "\nMissing source files in registry:",
    claim_registry[
        "SourceFile"
    ]
    .isna()
    .sum()
)


# ============================================================
# 31. Print Phase 2 frozen synthesis
# ============================================================

print("\n")
print("=" * 135)
print("PHASE 2 FINAL FROZEN SYNTHESIS")
print("=" * 135)


print(
    phase2_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 32. Print Phase 3 headline values
# ============================================================

print("\n")
print("=" * 135)
print("PHASE 3 FINAL FROZEN SYNTHESIS")
print("=" * 135)


phase3_display = (

    arch43[
        [
            "Outcome",
            "Mean_Core14_AttributionMass",
            "Mean_Core14_RenormCosine",
            "Mean_Core14_RankSpearman",
            "BELONG_MeanRank",
        ]
    ]
    .merge(

        perf43[
            [
                "Outcome",
                "Mean_Delta",
                "Pct_ExpandedBetter",
            ]
        ],

        on="Outcome",

        how="left",
    )
)


print(
    phase3_display
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print("\nAchievement-inclusive validation:")


phase3_achievement_display = (

    arch46[
        [
            "Outcome",
            "Mean_Core17_AttributionMass",
            "Mean_Core17_RenormCosine",
            "Mean_Core17_RankSpearman",
            "BELONG_MeanRank",
            "CoreAchievementShare",
            "ExpandedAchievementShare",
        ]
    ]
    .merge(

        perf46[
            [
                "Outcome",
                "Mean_Delta",
                "Pct_ExpandedBetter",
            ]
        ],

        on="Outcome",

        how="left",
    )
)


print(
    phase3_achievement_display
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 33. Print cross-phase manuscript argument
# ============================================================

print("\n")
print("=" * 135)
print("CROSS-PHASE HEADLINE ARGUMENT")
print("=" * 135)


print(
    headline_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 34. Output files
# ============================================================

print("\n")
print("=" * 135)
print("PHASE 5B SYNTHESIS COMPLETE")
print("=" * 135)


print(
    "\nCreated:"
)


for filename in [

    "final_frozen_claim_registry_phase1_to_phase4.csv",

    "final_frozen_source_manifest.csv",

    "phase2_frozen_synthesis_summary.csv",

    "cross_phase_headline_summary.csv",

]:

    print(
        " -",
        OUT_DIR
        /
        filename
    )


print(
    "\nNo new scientific analysis was introduced."
)


print(
    "The Phase 1–4 results are now consolidated for "
    "manuscript and figure construction."
)



FINAL FROZEN SOURCE AUDIT
p1_similarity                    FOUND | ..\results\similarity_framework_v2\similarity_framework_compact.csv
p2_pca                           FOUND | ..\results\international_heterogeneity\pca_explained_variance.csv
p2_cross_outcome                 FOUND | ..\results\international_heterogeneity\cross_outcome_distance_agreement.csv
p2_center_distance               FOUND | ..\results\international_heterogeneity\between_outcome_center_distances.csv
p2_distinctiveness               FOUND | ..\results\international_heterogeneity\system_aitchison_distinctiveness.csv
p2_macro_tests                   FOUND | ..\results\macro_context\macro_multivariate_tests.csv
p2_macro_dispersion              FOUND | ..\results\macro_context\macro_dispersion_tests.csv
p2_macro_coverage                FOUND | ..\results\macro_context\macro_metadata_coverage.csv
p2_cluster_robustness            FOUND | ..\results\cluster_interpretation\focal_cluster_solution_robustness.csv
p2_cluster

In [5]:
# ============================================================
# PHASE 5B PATCH — CORRECT EXTENDED-CONTEXT SYNTHESIS
# ============================================================
#
# NO NEW ANALYSIS.
#
# The prior synthesis counted all rows from
# repaired_high_priority_context_tests.csv.
#
# We need the FINAL adjusted tests only:
# added contextual validators after GDP + region adjustment.
# ============================================================

from pathlib import Path
import pandas as pd


RESULT_ROOT = Path("../results")

OUT_DIR = (
    RESULT_ROOT
    / "cross_phase_synthesis"
)


EXTENDED_PATH = (
    RESULT_ROOT
    / "extended_context_patterns"
    / "repaired_high_priority_context_tests.csv"
)


extended = pd.read_csv(
    EXTENDED_PATH
)


# ============================================================
# 1. Inspect exact test labels
# ============================================================

print("=" * 120)
print("EXTENDED-CONTEXT TEST TYPES")
print("=" * 120)

print(
    extended[
        "Test"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


print("\n")
print("=" * 120)
print("EXTENDED-CONTEXT RESULTS BY TEST TYPE")
print("=" * 120)

print(
    extended[
        [
            "Response",
            "Validator",
            "Test",
            "N_systems",
            "PartialR2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "Test",
            "FDR_Q",
            "PermutationP",
        ]
    )
    .round(5)
    .to_string(
        index=False
    )
)


# ============================================================
# 2. Identify adjusted tests
#
# Match labels containing both the idea of adjustment and
# GDP/region controls.
# ============================================================

test_text = (
    extended[
        "Test"
    ]
    .astype(str)
    .str.lower()
)


adjusted_mask = (

    (
        test_text.str.contains(
            "adjust",
            regex=False
        )
        |
        test_text.str.contains(
            "gdp",
            regex=False
        )
        |
        test_text.str.contains(
            "region",
            regex=False
        )
    )
)


adjusted = (
    extended[
        adjusted_mask
    ]
    .copy()
)


print("\n")
print("=" * 120)
print("FINAL ADJUSTED EXTENDED-CONTEXT TESTS")
print("=" * 120)

print(
    adjusted[
        [
            "Response",
            "Validator",
            "Test",
            "N_systems",
            "PartialR2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "FDR_Q",
            "PermutationP",
        ]
    )
    .round(5)
    .to_string(
        index=False
    )
)


print(
    "\nAdjusted tests:",
    len(adjusted)
)


print(
    "Adjusted tests surviving FDR:",
    int(
        (
            adjusted[
                "FDR_Q"
            ]
            <
            0.05
        )
        .sum()
    )
)


# ============================================================
# 3. Correct Phase-2 synthesis summary
# ============================================================

summary_path = (
    OUT_DIR
    / "phase2_frozen_synthesis_summary.csv"
)


summary = pd.read_csv(
    summary_path
)


mask = (
    summary[
        "Finding"
    ]
    ==
    "Extended-context tests surviving FDR"
)


summary.loc[
    mask,
    "Value"
] = (
    f"{int((adjusted['FDR_Q'] < 0.05).sum())}"
    "/"
    f"{len(adjusted)}"
)


summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 4. Save exact adjusted test subset
# ============================================================

adjusted.to_csv(
    OUT_DIR
    / "phase2_final_adjusted_extended_context_tests.csv",
    index=False
)


# ============================================================
# 5. Print corrected summary
# ============================================================

print("\n")
print("=" * 120)
print("CORRECTED PHASE 2 FROZEN SYNTHESIS")
print("=" * 120)

print(
    summary.to_string(
        index=False
    )
)


print("\nPhase 5B extended-context synthesis corrected.")

EXTENDED-CONTEXT TEST TYPES
Test
Unadjusted              6
AdjustedForGDPRegion    6


EXTENDED-CONTEXT RESULTS BY TEST TYPE
          Response               Validator                 Test  N_systems  PartialR2  PermutationP   FDR_Q
SubtypeReweighting                HDI_2018 AdjustedForGDPRegion         70    0.03226       0.01780 0.05339
SubtypeReweighting                GII_2018 AdjustedForGDPRegion         68    0.02452       0.07499 0.11248
SubtypeReweighting GovernmentEffectiveness AdjustedForGDPRegion         72    0.02039       0.14077 0.14077
    BaselineJoint5                HDI_2018 AdjustedForGDPRegion         69    0.02234       0.09398 0.19696
    BaselineJoint5                GII_2018 AdjustedForGDPRegion         67    0.02099       0.15097 0.19696
    BaselineJoint5 GovernmentEffectiveness AdjustedForGDPRegion         71    0.01875       0.19696 0.19696
    BaselineJoint5                HDI_2018           Unadjusted         69    0.06304       0.00020 0.00030
    Baselin

In [7]:
# ============================================================
# PHASE 5B PATCH v2 — EXACT EXTENDED-CONTEXT SELECTION
# ============================================================
#
# FIX:
# "Unadjusted" contains the substring "adjust", so the prior
# substring-based selector incorrectly retained all 12 rows.
#
# The exact final adjusted test label is:
#
#       AdjustedForGDPRegion
#
# NO NEW ANALYSIS.
# ============================================================

from pathlib import Path
import pandas as pd


RESULT_ROOT = Path("../results")

OUT_DIR = (
    RESULT_ROOT
    / "cross_phase_synthesis"
)


EXTENDED_PATH = (
    RESULT_ROOT
    / "extended_context_patterns"
    / "repaired_high_priority_context_tests.csv"
)


# ============================================================
# 1. Load frozen results
# ============================================================

extended = pd.read_csv(
    EXTENDED_PATH
)


# ============================================================
# 2. EXACT adjusted subset
# ============================================================

adjusted = (

    extended[
        extended[
            "Test"
        ]
        ==
        "AdjustedForGDPRegion"
    ]
    .copy()
)


assert len(
    adjusted
) == 6


assert (
    adjusted[
        "Test"
    ]
    ==
    "AdjustedForGDPRegion"
).all()


# ============================================================
# 3. FDR result
# ============================================================

n_fdr = int(
    (
        adjusted[
            "FDR_Q"
        ]
        <
        0.05
    )
    .sum()
)


assert n_fdr == 0


# ============================================================
# 4. Save authoritative adjusted subset
# ============================================================

adjusted.to_csv(
    OUT_DIR
    / "phase2_final_adjusted_extended_context_tests.csv",
    index=False
)


# ============================================================
# 5. Correct frozen Phase-2 synthesis summary
# ============================================================

summary_path = (
    OUT_DIR
    / "phase2_frozen_synthesis_summary.csv"
)


summary = pd.read_csv(
    summary_path
)


target = (
    summary[
        "Finding"
    ]
    ==
    "Extended-context tests surviving FDR"
)


if target.sum() != 1:

    raise RuntimeError(
        "Expected exactly one extended-context summary row."
    )


summary.loc[
    target,
    "Value"
] = (
    f"{n_fdr}/{len(adjusted)}"
)


summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 6. Print exact adjusted results
# ============================================================

print("\n")
print("=" * 120)
print("AUTHORITATIVE ADJUSTED EXTENDED-CONTEXT RESULTS")
print("=" * 120)


print(
    adjusted[
        [
            "Response",
            "Validator",
            "N_systems",
            "PartialR2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "FDR_Q",
            "PermutationP",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print("\nAdjusted tests:", len(adjusted))

print(
    "Adjusted tests surviving FDR:",
    n_fdr
)


# ============================================================
# 7. Print corrected synthesis
# ============================================================

print("\n")
print("=" * 120)
print("FINAL CORRECTED PHASE 2 FROZEN SYNTHESIS")
print("=" * 120)


print(
    summary.to_string(
        index=False
    )
)


print("\nPhase 2 synthesis is now corrected and frozen.")



AUTHORITATIVE ADJUSTED EXTENDED-CONTEXT RESULTS
          Response               Validator  N_systems  PartialR2  PermutationP   FDR_Q
SubtypeReweighting                HDI_2018         70    0.03226       0.01780 0.05339
SubtypeReweighting                GII_2018         68    0.02452       0.07499 0.11248
SubtypeReweighting GovernmentEffectiveness         72    0.02039       0.14077 0.14077
    BaselineJoint5                HDI_2018         69    0.02234       0.09398 0.19696
    BaselineJoint5                GII_2018         67    0.02099       0.15097 0.19696
    BaselineJoint5 GovernmentEffectiveness         71    0.01875       0.19696 0.19696

Adjusted tests: 6
Adjusted tests surviving FDR: 0


FINAL CORRECTED PHASE 2 FROZEN SYNTHESIS
                                             Finding                                            Value
    First 3 PCA components cumulative variance range                                      0.485–0.651
          Components required for 80% varia